In [2]:
import pandas as pd
df=pd.read_csv("/mnt/sda1/osint_intern/SA_Group/Raunak/data.csv")

In [3]:
import os
os.getcwd()

'/mnt/sda1/osint_intern/SA_Group/Raunak/Culture_Aware_Sentiment_Analysis/Implementation/Mistral'

In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.5.1+cu121
12.1


In [4]:
import bitsandbytes as bnb

In [5]:
print(bnb.__version__)

0.49.2


In [12]:
import sys
print(sys.executable)
!{sys.executable} -m pip install ollama

/mnt/sda1/osint_intern/miniconda3/envs/qwen/bin/python
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)


In [6]:
import sys
!{sys.executable} -m pip show ollama

Name: ollama
Version: 0.6.2
Summary: The official Python client for Ollama.
Home-page: https://ollama.com
Author: 
Author-email: hello@ollama.com
License-Expression: MIT
Location: /mnt/sda1/osint_intern/miniconda3/envs/qwen/lib/python3.12/site-packages
Requires: httpx, pydantic
Required-by: 


In [7]:
from ollama import Client

client = Client()

response = client.chat(
    model="mistral",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response["message"]["content"])

 Hello! How can I help you today? If you have any questions or need information on a specific topic, feel free to ask. I'm here to help!

Here are some examples of what I can assist with:

* Answering factual questions
* Providing explanations on various topics
* Suggesting interesting articles or resources
* Helping with homework or coursework
* Offering advice or strategies on various subjects

If you're looking for a friendly and knowledgeable companion to chat with, I'm always here for that too! Let me know if there's anything I can help you with.


In [ ]:
import os
import re
import json
import torch
import pandas as pd
from tqdm.auto import tqdm

# =====================================================
# PROMPT
# =====================================================

PROMPT_TEMPLATE = """
You are an expert sentiment classifier.

Your task is to classify sentiment by considering:
- cultural context
- regional practices
- social norms
- geographical sensitivity
- local interpretation

A sentence may express different sentiment in different locations.

STRICT RULES:
- Output ONLY valid JSON
- No explanation
- No markdown
- No extra text

SCHEMA:
{{
    "sentiment":"positive | neutral | negative"
}}

Location:
{location}

Text:
{text}

Return ONLY JSON.
"""

# =====================================================
# SETTINGS
# =====================================================

BATCH_SIZE = 100
PROGRESS_FILE = "Mistral_inference_progress.csv"
FINAL_FILE = "Mistral_all_predictions.csv"

# =====================================================
# LOAD PREVIOUS PROGRESS IF AVAILABLE
# =====================================================

if os.path.exists(PROGRESS_FILE):

    print(
        f"Found existing progress file: "
        f"{PROGRESS_FILE}"
    )

    df = pd.read_csv(
        PROGRESS_FILE
    )

    print(
        "Previous progress loaded."
    )

else:

    print(
        "No previous progress found."
    )

    if "Sentiment" not in df.columns:
        df["Sentiment"] = None

# =====================================================
# JSON PARSER
# =====================================================

def extract_sentiment(response):

    response = str(response).strip()

    try:

        match = re.search(
            r"\{.*?\}",
            response,
            re.DOTALL
        )

        if match:

            obj = json.loads(
                match.group()
            )

            sentiment = (
                obj["sentiment"]
                .strip()
                .lower()
            )

            if sentiment in [
                "positive",
                "neutral",
                "negative"
            ]:
                return sentiment

    except:
        pass

    text = response.lower()

    if "positive" in text:
        return "positive"

    if "neutral" in text:
        return "neutral"

    if "negative" in text:
        return "negative"

    return "PARSE_ERROR"

# =====================================================
# QWEN / GEMMA INFERENCE
# =====================================================
# =====================================================
# OLLAMA MISTRAL INFERENCE
# =====================================================

def classify_batch(batch_df):

    sentiments = []

    for _, row in batch_df.iterrows():

        location = (
            f"{row['State']} - "
            f"{row['Region']}"
        )

        prompt = PROMPT_TEMPLATE.format(
            location=location,
            text=str(row["Sentence"])
        )

        try:

            response = client.chat(

                model="mistral",

                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]

            )

            decoded = response["message"]["content"]

            sentiments.append(
                extract_sentiment(decoded)
            )

        except Exception as e:

            print(f"Error: {e}")

            sentiments.append("ERROR")

    return sentiments
# =====================================================
# DETERMINE STARTING POINT
# =====================================================

if "Sentiment" not in df.columns:
    df["Sentiment"] = None

completed_mask = (
    df["Sentiment"]
    .notna()
)

start_idx = int(
    completed_mask.sum()
)

print(
    f"Completed rows: "
    f"{start_idx}/{len(df)}"
)

# =====================================================
# MAIN PROCESSING LOOP
# =====================================================

for start in tqdm(
    range(start_idx, len(df), BATCH_SIZE),
    desc="Classifying"
):

    end = min(
        start + BATCH_SIZE,
        len(df)
    )

    batch_df = df.iloc[
        start:end
    ]

    preds = classify_batch(
        batch_df
    )

    df.loc[
        batch_df.index,
        "Sentiment"
    ] = preds

    # =====================================
    # SAVE AFTER EVERY BATCH
    # =====================================

    df.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved progress: "
        f"{end}/{len(df)}"
    )

# =====================================================
# FINAL SAVE
# =====================================================

df.to_csv(
    FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nSaved final file -> "
    f"{FINAL_FILE}"
)

# =====================================================
# SAVE REGION-WISE FILES
# =====================================================

for region in df["Region"].dropna().unique():

    region_df = df[
        df["Region"] == region
    ].copy()

    filename = (
        region.replace(
            " ",
            "_"
        )
        + "Mistral.csv"
    )

    region_df.to_csv(
        filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved -> {filename}"
    )

print(
    "\nFinished Successfully."
)


/mnt/sda1/osint_intern/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No previous progress found.
Completed rows: 0/11832


Classifying:   1%|█▎                                                                                                                                                           | 1/119 [02:22<4:40:26, 142.60s/it]

Saved progress: 100/11832


Classifying:   2%|██▋                                                                                                                                                          | 2/119 [04:46<4:39:23, 143.27s/it]

...

Saved progress: 11600/11832


Classifying:  98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍  | 117/119 [2:39:22<02:40, 80.41s/it]

Saved progress: 11700/11832


Classifying:  99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋ | 118/119 [2:40:43<01:20, 80.69s/it]

Saved progress: 11800/11832
